# GIẢI THÍCH THUẬT TOÁN & TƯ DUY SUY LUẬN — DỰ ÁN PHÂN LOẠI VĂN BẢN ESG TIẾNG VIỆT

> **Dành cho ai?** Dành cho bất kỳ ai — kể cả **người ngoài ngành AI / NLP**, giảng viên các chuyên ngành khác, hay người cần nắm nhanh **tinh thần cốt lõi** của dự án trước buổi bảo vệ — đều có thể đọc hiểu trọn vẹn **"chúng ta đang làm gì, tại sao làm như vậy, và mỗi thuật toán hoạt động ra sao"** trong ~10 phút.

--- 
## Tóm tắt tinh thần dự án 

Hãy tưởng tượng bạn cần xây dựng một **trợ lý AI đọc báo cáo doanh nghiệp tiếng Việt** và xếp từng câu vào 4 ngăn tủ:
1. **Môi trường (`Environment`):** nói về khí thải, điện năng, nước, rác thải, tái chế.
2. **Xã hội (`Social`):** nói về nhân viên, phúc lợi, an toàn lao động, đào tạo, cộng đồng.
3. **Quản trị (`Governance`):** nói về hội đồng quản trị, kiểm toán, quản lý rủi ro, minh bạch, chống tham nhũng.
4. **Không liên quan (`Irrelevant`):** câu kinh doanh thông thường (doanh thu, mở chi nhánh, tiêu đề...).

**Khó khăn thực tế:** Bạn có `27.365` câu đã được chuyên gia gắn nhãn sẵn, nhưng lại có tới **`54.487` câu mới chưa ai đọc và gắn nhãn** (thuê chuyên gia đọc hết tốn rất nhiều tiền và thời gian). Chưa kể, trong số câu đã gắn nhãn, tủ *Xã hội* quá đầy (`36.6%`) còn tủ *Môi trường* lại thiếu (`16.0%`).

**Ý tưởng giải quyết (Iterative Ensemble Pseudo-Labeling):**
```text
[1] Mời 7 chuyên gia AI khác nhau học bài sơ bộ trên dữ liệu gốc (Vòng 1)
        │
        ▼
[2] Đưa 54.487 câu CHƯA CÓ NHÃN cho cả 7 chuyên gia cùng đọc & BỎ PHIẾU
    → Chỉ giữ lại những câu >= 5/7 chuyên gia đồng ý VÀ độ tự tin rất cao (>= 94%)
        │
        ▼
[3] Ưu tiên chọn thêm các câu chắc chắn nhất cho những tủ đang thiếu (Cân bằng 4 lớp)
    → Xóa trí nhớ cũ, cho cả 7 chuyên gia HỌC LẠI TỪ ĐẦU trên tập bài học mới (Vòng 2 → Vòng 3)
        │
        ▼
[4] Khi đi thi thật (3.000 câu đề thi KHÓA KÍN): cả 7 chuyên gia cùng bỏ phiếu
    → Độ chính xác tăng từ 86.46% (chỉ học gốc) lên 90.09% Macro-F1 / 90.77% Accuracy!
```

---
## Phần 1 — Tại sao cần "Đề thi khóa kín" (Zero Data Leakage) & Cạm bẫy "Ôn lệch tủ"?

### 1.1. Ẩn dụ "Chống lộ đề thi" (`Zero Data Leakage`)
Trong nghiên cứu AI, một lỗi rất hay gặp là để câu trong đề thi bị lọt vào tài liệu ôn tập — giống như **học sinh học thuộc lòng đáp án đề thi trước khi vào phòng thi**, điểm số rất cao nhưng ra đời thật thì làm sai hết.

Vì vậy, ngay từ bước đầu (Bước B2), hệ thống quét sạch toàn bộ **`84.856` câu gốc** và tách rạch ròi làm 4 phần không bao giờ chạm mặt nhau:
- **Sách giáo khoa gốc (`train_pool.csv` — `24.628` câu):** dùng để dạy mô hình.
- **Đề thi thử (`val.csv` — `2.737` câu):** dùng để kiểm tra xem mô hình học đến epoch nào là tốt nhất thì dừng (tránh học vẹt).
- **Đề thi chính thức KHÓA KÍN (`test_clean.csv` — `3.000` câu):** **tuyệt đối không mở ra** trong lúc dạy hay chọn mô hình; chỉ mở để chấm điểm khách quan cuối mỗi vòng. Kiểm toán xác nhận **`0` câu trùng** giữa tập này với các tập còn lại.
- **Kho bài tập chưa có đáp án (`unlabeled_clean.csv` — `54.487` câu):** nguồn dữ liệu để 7 chuyên gia AI tự bỏ phiếu khai thác thêm.

### 1.2. Ẩn dụ "Ôn lệch tủ" (`Mất cân bằng lớp` & `Prior Shift`) và tại sao dùng `Macro-F1` thay vì `Accuracy`?
- Trong sách giáo khoa gốc (`train_pool`), lớp **Xã hội (`36.64%`)** nhiều gấp **2.29 lần** lớp **Môi trường (`16.02%`)**.
- Nhưng trong đề thi chính thức (`test`), tỷ lệ 4 lớp gần cân bằng nhau (**Môi trường tăng lên `27.33%`** — chênh lệch `+11.31%`!). Hiện tượng *tỷ lệ đề thi khác tỷ lệ tài liệu ôn* gọi là **Prior Probability Shift**.
- Nếu ta chỉ chấm theo **Accuracy** (tổng số câu đúng / tổng số câu), một mô hình "lười biếng" cứ đoán bừa mọi câu là lớp đa số vẫn kiếm được kha khá điểm, nhưng **bỏ sót sạch lớp thiểu số**! Ngược lại, **Macro-F1** tính điểm riêng cho từng ngăn tủ rồi lấy trung bình cộng 4 ngăn — hễ bỏ quên bất kỳ ngăn nào là điểm tụt ngay lập tức.

Sự khác biệt giữa **Accuracy** và **Macro-F1**!

In [1]:
# Minh họa trực quan: Tại sao Macro-F1 công bằng hơn Accuracy khi dữ liệu lệch lớp?
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# Giả sử đề thi có 100 câu: 70 câu lớp Social (2), còn 3 lớp kia mỗi lớp chỉ 10 câu
y_true = np.array([2]*70 + [0]*10 + [1]*10 + [3]*10)

# Thí sinh A ("đoán lụi"): nhắm mắt điền 100% đáp án là Social (2)
y_pred_lazy = np.array([2]*100)

# Thí sinh B ("học đều"): làm đúng 70% ở MỌI lớp (kể cả lớp ít câu)
y_pred_fair = np.array(
    [2]*49 + [0]*7 + [1]*7 + [3]*7 +   # 70 câu Social: đúng 49
    [0]*7 + [2]*3 +                     # 10 câu Irrelevant: đúng 7
    [1]*7 + [2]*3 +                     # 10 câu Environment: đúng 7
    [3]*7 + [2]*3                       # 10 câu Governance: đúng 7
)

print("=== SO SÁNH THƯỚC ĐO ACCURACY vs MACRO-F1 ===")
print(f"Thí sinh A (đoán lụi 100% lớp đa số): Accuracy = {accuracy_score(y_true, y_pred_lazy)*100:5.1f}% | Macro-F1 = {f1_score(y_true, y_pred_lazy, average='macro')*100:5.1f}%  <-- Bị vạch trần ngay!")
print(f"Thí sinh B (hiểu đều cả 4 lớp)     : Accuracy = {accuracy_score(y_true, y_pred_fair)*100:5.1f}% | Macro-F1 = {f1_score(y_true, y_pred_fair, average='macro')*100:5.1f}%  <-- Phản ánh đúng thực lực!")
print("\n=> Kết luận: Chính vì lý do này, toàn bộ dự án chọn checkpoint tốt nhất bằng Macro-F1 chứ không chọn theo Accuracy.")

=== SO SÁNH THƯỚC ĐO ACCURACY vs MACRO-F1 ===
Thí sinh A (đoán lụi 100% lớp đa số): Accuracy =  70.0% | Macro-F1 =  20.6%  <-- Bị vạch trần ngay!
Thí sinh B (hiểu đều cả 4 lớp)     : Accuracy =  70.0% | Macro-F1 =  62.9%  <-- Phản ánh đúng thực lực!

=> Kết luận: Chính vì lý do này, toàn bộ dự án chọn checkpoint tốt nhất bằng Macro-F1 chứ không chọn theo Accuracy.


---
## Phần 2 — Tại sao dùng "Hội đồng 7 chuyên gia AI" thay vì chỉ 1 mô hình?

Nếu bạn chỉ hỏi ý kiến 1 chuyên gia duy nhất, đôi khi họ có điểm mù (ví dụ quen tiếng Việt thuần nhưng lúng túng khi gặp chữ viết tắt tiếng Anh như *GRI, Net Zero, SHEQ*, hoặc ngược lại). Lý thuyết học kết hợp (*Dietterich, 2000*) chứng minh rằng khi kết hợp **nhiều chuyên gia có nền tảng học khác nhau** (*Ensemble Diversity*), lỗi sai ngẫu nhiên của từng cá nhân sẽ tự triệt tiêu lẫn nhau!

| Chuyên gia AI | Gốc đào tạo | Ví von dễ hiểu | Cách đọc văn bản đầu vào |
| :--- | :--- | :--- | :--- |
| **1. PhoBERT-base** | Tiếng Việt (VinAI) | Biên tập viên báo chính thống tiếng Việt, đọc theo **từ ghép** | **Cần `pyvi` nối từ ghép** (`khí_nhà_kính`) |
| **2. Vi-ELECTRA** | Tiếng Việt (HUST) | Người soát lỗi chính tả nhạy bén (học cách phát hiện từ bị tráo đổi) | Đọc câu tự nhiên (subword) |
| **3. ViSoBERT** | Tiếng Việt (UIT) | Chuyên gia đọc tin tức mạng & ngôn ngữ hiện đại, gọn nhẹ | Đọc câu tự nhiên (SentencePiece) |
| **4. ViDeBERTa-base** | Tiếng Việt (FPT AI) | Nhà phân tích cẩn trọng, chú ý mạnh vị trí từ trong câu dài | Đọc câu tự nhiên (SentencePiece) |
| **5. mBERT-cased** | Đa ngôn ngữ (Google) | Thông dịch viên biết 104 thứ tiếng, giỏi câu pha tiếng Anh | Đọc câu tự nhiên (WordPiece) |
| **6. DistilBERT-m** | Đa ngôn ngữ (rút gọn 6 lớp) | Trợ lý song ngữ siêu nhanh, phản xạ cực gọn, ít học vẹt | Đọc câu tự nhiên (WordPiece) |
| **7. RoBERTa-base** | Tiếng Anh (Meta/FB) | Chuyên gia ngoại ngữ bổ sung góc nhìn hoàn toàn khác biệt | Đọc câu tự nhiên (Byte-BPE) |

### Minh họa thực tế: Vì sao chỉ riêng `PhoBERT` cần nối từ bằng `pyvi`?
Đầu vào mà `PhoBERT` nhìn thấy so với 6 mô hình còn lại:

In [3]:
# Minh họa cơ chế Tiền xử lý đồng nhất theo kiến trúc (Architecture-Aligned Preprocessing)
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import VietnameseESGPreprocessor

sample_sentence = "Tập đoàn đầu tư năng lượng tái tạo nhằm giảm phát thải khí nhà kính theo chuẩn ESG quốc tế."

prep_phobert = VietnameseESGPreprocessor(use_word_segmenter=True)
prep_others  = VietnameseESGPreprocessor(use_word_segmenter=False)

print("Câu gốc ban đầu             :", sample_sentence)
print("Đầu vào cho PhoBERT (pyvi)  :", prep_phobert.segment_text(sample_sentence))
print("Đầu vào cho 6 mô hình còn lại:", prep_others.segment_text(sample_sentence))
print("\n=> Lưu ý: PhoBERT được học từ lúc sơ khai bằng từ ghép nối dấu gạch dưới (_). Nếu quên tách từ cho PhoBERT, nó sẽ đọc sai nghĩa từ ghép; ngược lại, nếu tách từ cho mBERT/DistilBERT thì lại làm rối bộ từ vựng của chúng!")

Câu gốc ban đầu             : Tập đoàn đầu tư năng lượng tái tạo nhằm giảm phát thải khí nhà kính theo chuẩn ESG quốc tế.
Đầu vào cho PhoBERT (pyvi)  : Tập_đoàn đầu_tư năng_lượng tái_tạo nhằm giảm_phát thải khí nhà_kính theo chuẩn ESG quốc_tế .
Đầu vào cho 6 mô hình còn lại: Tập đoàn đầu tư năng lượng tái tạo nhằm giảm phát thải khí nhà kính theo chuẩn ESG quốc tế.

=> Lưu ý: PhoBERT được học từ lúc sơ khai bằng từ ghép nối dấu gạch dưới (_). Nếu quên tách từ cho PhoBERT, nó sẽ đọc sai nghĩa từ ghép; ngược lại, nếu tách từ cho mBERT/DistilBERT thì lại làm rối bộ từ vựng của chúng!


---
## Phần 3 — Trái tim của thuật toán: Hội đồng 7 giám khảo tự chấm 54.487 câu không nhãn (`Iterative Ensemble Pseudo-Labeling`)

Đây là câu hỏi trọng tâm nhất mà hội đồng sẽ hỏi: **"Làm sao đảm bảo nhãn giả do AI tự sinh ra không bị sai rác rồi dạy hư ngược lại chính AI?"**

Hệ thống sử dụng **5 lớp bảo vệ chặt chẽ liên hoàn**:
1. **Lớp bảo vệ 1 — Cổng chất lượng (`Quality Gate`, ngưỡng `Val Macro-F1 >= 50%`):** Trước khi cho 7 giám khảo chấm bài, ta xem điểm thi thử của từng người. Ai thi thử dưới 50% (ví dụ bị suy biến như `ViDeBERTa-base` ở Vòng 1) sẽ bị **tước quyền biểu quyết (`trọng số = 0`)** ngay lập tức!
2. **Lớp bảo vệ 2 — Trung bình xác suất có trọng số (`Weighted Soft Voting`):** Giám khảo nào thi thử điểm cao hơn sẽ có tiếng nói nặng ký hơn một chút khi cộng gộp xác suất dự đoán của cả hội đồng.
3. **Lớp bảo vệ 3 — Bộ 3 điều kiện lọc khắc nghiệt (phải qua CẢ 3 mới nhận nhãn giả):**
   - **Độ tự tin cao (`Confidence >= 94%` ở Vòng 1, `>= 91%` ở Vòng 2):** Đáp án được chọn phải có xác suất áp đảo.
   - **Khoảng cách an toàn (`Margin >= 50%` ở Vòng 1, `>= 45%` ở Vòng 2):** Hiệu giữa xác suất hạng 1 và hạng 2 phải cách xa nhau — loại bỏ ngay các câu lưỡng lự giữa 2 lớp (ví dụ 52% Social vs 45% Governance).
   - **Đồng thuận số đông (`Hard Consensus >= 5/7` giám khảo):** Ít nhất 5 trên 7 mô hình phải độc lập chọn đúng cùng một đáp án.
4. **Lớp bảo vệ 4 — Cân bằng lớp có chọn lọc (`Selective Class-Balanced Combining`):** Thay vì đổ ồ ạt mọi nhãn giả vào tập huấn luyện (sẽ làm lớp đa số càng phình to), hệ thống sắp xếp các câu đạt chuẩn theo độ tự tin giảm dần và **chỉ lấy vừa đủ lên chỉ tiêu cân bằng** (`9.500` mẫu/lớp ở Vòng 1, `10.500` mẫu/lớp ở Vòng 2). Kết quả: tỷ số mất cân bằng `IR` giảm đẹp từ **`2.29 → 1.61 → 1.35`**!
5. **Lớp bảo vệ 5 — Xóa trí nhớ học lại từ đầu (`Re-init from Pretrained`):** Trước khi vào Vòng 2 và Vòng 3, tất cả 7 mô hình đều được khởi tạo sạch lại từ trọng số gốc (không huấn luyện tiếp từ checkpoint cũ) để không bị "ám ảnh" bởi những sai số nhỏ của vòng trước.

Mô phỏng dưới đây để xem tận mắt cách thuật toán chấm điểm và ra quyết định **NHẬN** hay **LOẠI** trên 3 tình huống thực tế!

In [4]:
# Mô phỏng từng bước thuật toán Quality Gate + Weighted Soft Voting + 3 Bộ lọc Pseudo-Labeling
import numpy as np
import pandas as pd

LABELS = ["Irrelevant", "Environment", "Social", "Governance"]
MODELS = ["PhoBERT", "Vi-ELECTRA", "mBERT", "DistilBERT-m", "ViSoBERT", "RoBERTa", "ViDeBERTa"]

# Giả sử điểm thi thử (Val Macro-F1) ở Vòng 1 — trong đó ViDeBERTa bị lỗi chỉ đạt 0.134 (< 0.50)
val_f1_scores = np.array([0.896, 0.880, 0.881, 0.861, 0.845, 0.810, 0.134])

# BƯỚC 1: Áp dụng Quality Gate (gamma = 0.50)
mask = (val_f1_scores >= 0.50).astype(float)
weights = (val_f1_scores * mask) / np.sum(val_f1_scores * mask)
print("=== BƯỚC 1: QUALITY GATE (NGƯỠNG VAL F1 >= 0.50) ===")
for m, s, w in zip(MODELS, val_f1_scores, weights):
    status = "ĐƯỢC BỎ PHIẾU" if w > 0 else "BỊ LOẠI (TRỌNG SỐ = 0)"
    print(f"  {m:13s} | Val F1 = {s:.3f} | Trọng số = {w:.4f} | {status}")

def evaluate_candidate(sentence_name, model_probs, tau=0.94, delta=0.50, min_votes=5):
    # BƯỚC 2: Weighted Soft Voting (chỉ tính các mô hình qua Quality Gate)
    ens_prob = np.sum(model_probs * weights[:, None], axis=0)
    pred_id = int(np.argmax(ens_prob))
    sorted_p = np.sort(ens_prob)
    conf = float(sorted_p[-1])
    margin = float(sorted_p[-1] - sorted_p[-2])
    # Đếm số phiếu đồng thuận trong các mô hình hợp lệ
    hard_votes = np.argmax(model_probs, axis=1)
    valid_votes = int(np.sum((hard_votes == pred_id) & (weights > 0)))
    passed = (conf >= tau) and (margin >= delta) and (valid_votes >= min_votes)
    print(f"\n--- {sentence_name} ---")
    print(f"  Xác suất hội đồng : Irrel={ens_prob[0]:.3f}, Env={ens_prob[1]:.3f}, Soc={ens_prob[2]:.3f}, Gov={ens_prob[3]:.3f}")
    print(f"  Nhãn ứng viên     : {LABELS[pred_id]} | Conf={conf:.3f} (cần >={tau}) | Margin={margin:.3f} (cần >={delta}) | Phiếu={valid_votes}/6 hợp lệ (cần >={min_votes})")
    print(f"  => QUYẾT ĐỊNH     : {'[CHẤP NHẬN LÀM NHÃN GIẢ SẠCH]' if passed else '[LOẠI BỎ — ĐỂ LẠI XÉT SAU HOẶC BỎ QUA]'}")

# Câu A: Rõ ràng thuộc Environment (cả 6 mô hình hợp lệ đồng thuận >96%, riêng ViDeBERTa lỗi bầu Social nhưng bị khóa trọng số=0)
probs_A = np.array([
    [0.01, 0.97, 0.01, 0.01],
    [0.01, 0.98, 0.00, 0.01],
    [0.02, 0.96, 0.01, 0.01],
    [0.01, 0.97, 0.01, 0.01],
    [0.02, 0.95, 0.02, 0.01],
    [0.01, 0.96, 0.02, 0.01],
    [0.00, 0.00, 1.00, 0.00],  # ViDeBERTa bị collapse -> đã bị Quality Gate loại!
])

# Câu B: Câu giao thoa khó (Hội đồng quản trị họp về bình đẳng giới -> 3 mô hình chọn Social, 3 mô hình chọn Governance)
probs_B = np.array([
    [0.02, 0.01, 0.58, 0.39],
    [0.01, 0.01, 0.42, 0.56],
    [0.03, 0.01, 0.54, 0.42],
    [0.02, 0.01, 0.45, 0.52],
    [0.02, 0.01, 0.61, 0.36],
    [0.02, 0.01, 0.40, 0.57],
    [0.00, 0.00, 1.00, 0.00],
])

evaluate_candidate("Câu A (Môi trường rõ ràng — dù 1 mô hình lỗi)", probs_A)
evaluate_candidate("Câu B (Biên giao thoa Social <-> Governance)", probs_B)

=== BƯỚC 1: QUALITY GATE (NGƯỠNG VAL F1 >= 0.50) ===
  PhoBERT       | Val F1 = 0.896 | Trọng số = 0.1732 | ĐƯỢC BỎ PHIẾU
  Vi-ELECTRA    | Val F1 = 0.880 | Trọng số = 0.1701 | ĐƯỢC BỎ PHIẾU
  mBERT         | Val F1 = 0.881 | Trọng số = 0.1703 | ĐƯỢC BỎ PHIẾU
  DistilBERT-m  | Val F1 = 0.861 | Trọng số = 0.1664 | ĐƯỢC BỎ PHIẾU
  ViSoBERT      | Val F1 = 0.845 | Trọng số = 0.1633 | ĐƯỢC BỎ PHIẾU
  RoBERTa       | Val F1 = 0.810 | Trọng số = 0.1566 | ĐƯỢC BỎ PHIẾU
  ViDeBERTa     | Val F1 = 0.134 | Trọng số = 0.0000 | BỊ LOẠI (TRỌNG SỐ = 0)

--- Câu A (Môi trường rõ ràng — dù 1 mô hình lỗi) ---
  Xác suất hội đồng : Irrel=0.013, Env=0.965, Soc=0.011, Gov=0.010
  Nhãn ứng viên     : Environment | Conf=0.965 (cần >=0.94) | Margin=0.952 (cần >=0.5) | Phiếu=6/6 hợp lệ (cần >=5)
  => QUYẾT ĐỊNH     : [CHẤP NHẬN LÀM NHÃN GIẢ SẠCH]

--- Câu B (Biên giao thoa Social <-> Governance) ---
  Xác suất hội đồng : Irrel=0.020, Env=0.010, Soc=0.501, Gov=0.469
  Nhãn ứng viên     : Social | Conf=0.501 (c

### Kết quả thực tế trên 54.487 câu không nhãn của dự án
Nhờ 5 lớp bảo vệ trên, qua 2 vòng lọc hệ thống đã khai thác an toàn **`32.903 / 54.487` câu (`60.4%`)** — riêng Vòng 2 có tới **`97.52%` câu đạt đồng thuận tuyệt đối 7/7 mô hình** — và chủ động bỏ qua `21.584` câu mập mờ (`39.6%`), đồng thời cân bằng lại 4 ngăn tủ gần như hoàn hảo (`IR: 2.29 → 1.35`):

![Biểu đồ phễu khai thác dữ liệu không nhãn](../workspace_outputs/figures/B6_pseudo_round2/B6_01_bieu_do_pheu_unlabeled.png)

![Phân bố tập huấn luyện cuối cùng sau 2 vòng Pseudo-Labeling](../workspace_outputs/figures/B6_pseudo_round2/B6_02_phan_bo_du_lieu_3_tang.png)

---
## Phần 4 — Chứng minh không phải "ăn may": Đối chứng (Ablation Study) & Kiểm định McNemar

Một câu hỏi phản biện rất hay của giảng viên: *"Ở Vòng 1 bạn chỉ cho học 10.000 câu, sang Vòng 3 học 39.248 câu thì đương nhiên cao hơn rồi — làm sao biết nhãn giả thực sự có ích chứ không phải chỉ do đưa hết 24.628 câu gốc vào?"*

### 4.1. Câu trả lời bằng Thực nghiệm Đối chứng (`Ablation Baseline`)
Chúng ta đã chạy riêng một nhóm đối chứng (**Baseline**): cho 3 mô hình tiếng Việt chủ lực (`PhoBERT-base`, `Vi-ELECTRA`, `ViSoBERT`) học **100% toàn bộ `24.628` câu gốc** (KHÔNG dùng bất kỳ câu nhãn giả nào).
- **Kết quả:** Cả 3/3 mô hình ở Vòng 3 đều **đánh bại chính bản Baseline 100% dữ liệu gốc của nó** (`PhoBERT +1.67%`, `Vi-ELECTRA +1.50%`, `ViSoBERT +4.56%`), và **Final Ensemble 7M** vượt Baseline mạnh nhất tới **`+3.63%` Macro-F1 (`86.46% → 90.09%`)**!

![Tiến trình 3 vòng so với Baseline và Final Ensemble](../workspace_outputs/figures/B8_final_report/B8_01_tien_trinh_3_vong_va_ablation.png)

### 4.2. Câu trả lời bằng Kiểm định thống kê McNemar (Không phải chênh lệch ngẫu nhiên)
Hiểu kiểm định **McNemar** cực kỳ đơn giản qua ẩn dụ **2 thí sinh cùng làm 3.000 câu trắc nghiệm**:
- Những câu cả 2 cùng đúng hoặc cả 2 cùng sai → hòa, không xét.
- Chỉ đếm những câu **1 người đúng, 1 người sai**:
  - `Equal Soft (7M)` vs `DistilBERT-m`: Ensemble sửa đúng **`99` câu** mà DistilBERT sai, chỉ làm sai **`67` câu** mà DistilBERT đúng (thắng ròng **`+32` câu**).
  - Xác suất để sự chênh lệch `99 vs 67` xảy ra do tung đồng xu ngẫu nhiên là **`p = 0.0161` (`1.61% < 5%`)** → Khẳng định **có ý nghĩa thống kê**!

Chạy cell dưới đây để tự kiểm chứng trực tiếp phép tính McNemar từ số liệu thật của dự án:

In [ ]:
# Tự tính trực tiếp Kiểm định McNemar (hiệu chỉnh liên tục Edwards) trên số liệu thực nghiệm
from scipy.stats import chi2

def mcnemar_explain(name_a, name_b, n10_a_correct, n01_b_correct):
    chi2_stat = ((abs(n10_a_correct - n01_b_correct) - 1.0) ** 2) / (n10_a_correct + n01_b_correct)
    p_val = float(1.0 - chi2.cdf(chi2_stat, df=1))
    verdict = "CÓ Ý NGHĨA THỐNG KÊ (p < 0.05) — không phải ngẫu nhiên!" if p_val < 0.05 else "Chưa đủ bằng chứng thống kê"
    print(f"So sánh [{name_a}] vs [{name_b}]:")
    print(f"  - {name_a} sửa đúng (B sai): {n10_a_correct} câu | {name_a} sai (B đúng): {n01_b_correct} câu | Chênh lệch ròng: +{n10_a_correct - n01_b_correct} câu")
    print(f"  - Thống kê chi2 = {chi2_stat:.4f} | p-value = {p_val:.6f} => {verdict}\n")

mcnemar_explain("Equal Soft (7M)", "Best Single (DistilBERT-m)", n10_a_correct=99, n01_b_correct=67)
mcnemar_explain("Equal Soft (7M)", "Best Baseline (PhoBERT-base)", n10_a_correct=162, n01_b_correct=73)

---
## Phần 5 — Hiệu chuẩn xác suất (`ECE`) & Đường biên Pareto (Ý nghĩa ứng dụng thực tế)

### 5.1. Ẩn dụ "Dự báo thời tiết" giải thích Hiệu chuẩn xác suất (`Calibration / ECE`)
Một đài dự báo thời tiết được gọi là **trung thực (hiệu chuẩn tốt)** khi: trong 100 ngày đài báo *"70% có mưa"*, thực tế có đúng khoảng 70 ngày mưa. Nếu đài báo *"70% mưa"* mà thực tế chỉ có 30 ngày mưa, đài đó đang **quá tự tin (`over-confident`)**!

- **Mô hình đơn lẻ (`DistilBERT-m`, `ECE = 0.0825`):** thường bị quá tự tin — ở vùng tự tin `50%–60%`, độ chính xác thực tế chỉ có `30%`.
- **Hội đồng 7 mô hình (`Equal Soft 7M`, `ECE = 0.0226`):** giảm sai số lệch hiệu chuẩn **gần 4 lần**! Khi hội đồng báo tự tin `>= 90%` (`2.316 / 3.000` câu = `77.2%` dữ liệu), tỷ lệ đoán đúng thực tế đạt **`97.41%`**!
- **Giá trị thực tiễn (Human-in-the-Loop):** Doanh nghiệp chỉ cần cho AI tự duyệt `77.2%` câu có độ tự tin `>= 90%`, và chỉ thuê chuyên gia đọc lại `22.8%` câu dưới ngưỡng `90%` → **tiết kiệm gần 4/5 chi phí nhân sự** mà vẫn giữ độ chính xác trên 97%!

![Biểu đồ hiệu chuẩn xác suất Reliability Diagram](../workspace_outputs/figures/B8_final_report/B8_06_calibration_reliability.png)

### 5.2. Ẩn dụ "Chọn phương tiện di chuyển" giải thích Đường biên Pareto
Khi triển khai thực tế, không có một lựa chọn nào vừa nhanh nhất vừa chính xác nhất — mà có **3 điểm tối ưu Pareto** tùy túi tiền và bài toán:
1. **Xe máy đi phố siêu nhanh (`DistilBERT-m`):** `0.324 ms/câu` (`3.085` câu/giây), chỉ tốn `2.96 GB` VRAM, đạt **`89.03%` Macro-F1** (giữ 98.8% sức mạnh của cả hội đồng 7 mô hình nhưng nhanh hơn **13.2 lần**!).
2. **Xe hơi cân bằng (`Top-3 Weighted`):** `1.54 ms/câu` (`650` câu/giây), **`89.08%` Macro-F1**, `ECE = 0.0589`.
3. **Hội đồng kiểm toán chuẩn xác nhất (`Equal Soft 7M`):** `4.27 ms/câu` (`234` câu/giây — đọc xong cả cuốn báo cáo `10.000` câu chỉ trong **43 giây**!), đạt **`90.09%` Macro-F1 / `90.77%` Acc / `ECE = 0.0226`**.

![Đánh đổi Hiệu năng và Độ trễ trên đường biên Pareto](../workspace_outputs/figures/B8_final_report/B8_05_tradeoff_hieu_nang_va_toc_do.png)

---
## Phần 6 — Giải thích sự cố kỹ thuật `ViDeBERTa-base` bằng ẩn dụ "Cân tiểu ly"

Trong Vòng 1, mô hình `ViDeBERTa-base` bị đóng băng hoàn toàn (đoán 100% lớp Social, `Macro-F1 = 11.04%`). Nguyên nhân gốc rễ cực kỳ thú vị và có thể giải thích trong 1 câu cho người ngoài ngành:

> **Ẩn dụ Cân tiểu ly:** Hãy tưởng tượng bạn dùng một chiếc cân chỉ chia vạch lớn nhất là **`0.001` gam** (tương đương độ chính xác số thực `float16`). Mỗi lần học, thuật toán AdamW thêm vào trọng số một lượng rất nhỏ là **`0.000015` gam** (`lr = 1.5e-5`). Vì lượng thêm vào nhỏ hơn vạch chia nhỏ nhất của chiếc cân, **kim cân cứ đứng im mãi ở số cũ** — mô hình không thể học thêm bất kỳ thứ gì!

Chạy cell Python 5 dòng dưới đây để chứng kiến tận mắt hiện tượng `float16 underflow` ngay trên máy của bạn!

In [ ]:
# Chứng minh thực nghiệm lỗi đóng băng trọng số float16 (nguyên nhân ViDeBERTa collapse ở Vòng 1)
import torch

step_update = 1.5e-5  # Bước cập nhật học nhỏ (learning rate của ViDeBERTa)

w_fp16 = torch.tensor(1.0, dtype=torch.float16)
w_fp32 = torch.tensor(1.0, dtype=torch.float32)

# Thử cập nhật trọng số 1.000 lần liên tiếp!
for _ in range(1000):
    w_fp16 = w_fp16 + torch.tensor(step_update, dtype=torch.float16)
    w_fp32 = w_fp32 + torch.tensor(step_update, dtype=torch.float32)

print(f"Sau 1.000 bước cập nhật (+1.5e-5 mỗi bước, kỳ vọng tăng +0.0150):")
print(f"  - Trọng số nạp thuần float16 (lỗi mặc định Transformers 5.24): 1.0 -> {w_fp16.item():.6f}  (ĐÓNG BĂNG HOÀN TOÀN!)")
print(f"  - Trọng số ép master FP32 (bản vá dtype=torch.float32)       : 1.0 -> {w_fp32.item():.6f}  (HỌC BÌNH THƯỜNG!)")

---
## Phần 7 — Bộ 8 Câu hỏi Phản biện Hội đồng thường gặp & Gợi ý Trả lời ngắn gọn

| STT | Câu hỏi có thể gặp từ Giảng viên / Hội đồng | Gợi ý trả lời chuẩn xác, bám sát số liệu thực nghiệm |
| :-: | :--- | :--- |
| **1** | *Tại sao không dùng luôn 100% tập `train_pool` gốc ở Vòng 1 thay vì chia 7 tập con 10k?* | Chia 7 tập con khác nhau giúp tạo **độ đa dạng lỗi** ngay từ vòng đầu (mô hình không mắc cùng một lỗi giống hệt nhau khi bỏ phiếu). Nhóm cũng đã chạy đối chứng **Ablation Baseline (100% `train_pool`)** và chứng minh cả 3/3 mô hình ở Vòng 3 đều vượt Baseline từ `+1.50%` đến `+4.56%` Macro-F1. |
| **2** | *Làm sao chắc chắn tập kiểm thử (`test.csv`) không bị rò rỉ dữ liệu?* | Bước B2 đã kiểm toán bằng khóa chuẩn hóa Unicode NFC trên cả `84.856` câu, xác nhận **`0` câu trùng** giữa `test.csv` với `train.csv` và `overall_data.csv`. Ngoài ra, checkpoint và trọng số Ensemble chỉ dùng `val.csv` (`2.737` câu tách riêng từ train), khóa kín `test.csv`. |
| **3** | *Vì sao ở Vòng 3 `DistilBERT-m` (6 lớp) lại cao điểm Test hơn `PhoBERT-base` (12 lớp)?* | Trên `val.csv`, `PhoBERT-base` vẫn cao nhất (`91.10%`), nhưng khoảng cách `Val − Test` của `PhoBERT` là `+2.97%`, còn `DistilBERT-m` chỉ `+0.34%` và có `ECE = 0.0825` thấp nhất nhóm đơn lẻ (ít quá tự tin). Ngoài ra, `DistilBERT-m` đa ngôn ngữ xử lý mượt các câu báo cáo ESG có xen thuật ngữ tiếng Anh mà không phụ thuộc bộ tách từ `pyvi`. |
| **4** | *Vì sao `Equal Soft (7M)` (`90.09%`) lại nhỉnh hơn `Weighted Soft (7M)` (`90.04%`)?* | Vì trọng số `Weighted` tính theo `Val Macro-F1` nên vô tình dồn trọng số nhiều hơn cho `PhoBERT`/`Vi-ELECTRA` (có độ chênh Val–Test lớn hơn) và giảm nhẹ tỷ trọng của `DistilBERT-m`, `RoBERTa`, `ViDeBERTa` (những mô hình bổ trợ rất tốt trên Test). Chia đều `1/7` giúp giảm thiên vị. |
| **5** | *Tại sao Vòng 2 có mô hình như `PhoBERT` giảm nhẹ `-0.54%` so với Vòng 1?* | Do ở Vòng 1 ngưỡng `0.94` rất chặt khiến lớp `Irrelevant` chỉ thêm được `592` câu (tỷ trọng tạm tụt xuống `17.2%`). Khi Vòng 2 bổ sung thêm `+1.844` câu `Irrelevant` (đưa tỷ trọng về `19.74%`, khớp `20.00%` của Test), ở Vòng 3 cả **7/7 mô hình** đều bật tăng rõ rệt (`PhoBERT` tăng `+2.24%` lên `88.13%`). |
| **6** | *Dùng 7 mô hình Ensemble (`988M` tham số) ngoài thực tế có nặng quá không?* | Trên 1 GPU RTX 5070, Ensemble 7M vẫn xử lý `234` câu/giây (`10.000` câu báo cáo mất chỉ ~43 giây) và giảm `ECE` xuống `0.0226` rất thích hợp cho kiểm toán theo lô. Nếu cần API thời gian thực gọn nhẹ, đường biên Pareto chỉ rõ chọn **`DistilBERT-m (Run 3)`** (`89.03%` Macro-F1, `0.324 ms/câu`, chỉ `2.96 GB` VRAM). |
| **7** | *Tại sao vẫn còn `9.23%` (`277` câu) đoán sai? Có thể khắc phục không?* | `82.31%` lỗi nằm ở 4 cặp biên (`Irrelevant ↔ Governance` và `Social ↔ Governance`). Phân tích Top-15 lỗi tự tin `>99.7%` cho thấy 3 nguyên nhân: (1) dấu hiệu nhiễu nhãn trong `test.csv` gốc (`row 2474, 413, 185, 1022`), (2) nhiều câu thực chất mang **đa nhãn** (HĐQT họp về nhân sự/DEI), và (3) thuật ngữ kế toán thuần túy. Hướng cải thiện là chuyển sang **Multi-Label ESG** và làm sạch nhãn bằng **Cleanlab**. |
| **8** | *Tại sao ở Bước B5/B6 lại đổi biểu đồ mật độ KDE sang đường cong `1 - ECDF`?* | Vì các mô hình Transformer sau fine-tune có xác suất tự tin dồn sát `0.99–1.00`, biểu đồ KDE dựng cột rất cao ở `1.0` làm ép phẳng toàn bộ vùng `0.40–0.90`. Đường cong tích lũy ngược `1 - ECDF` (`P(Conf >= x)`) giúp nhìn tách bạch rõ tỷ lệ vượt ngưỡng của từng mô hình trên toàn bộ trục xác suất. |